### Re-tokenize and re-BLEU
Re-tokenize (change the token length)

In [ ]:
### import libraries

import pandas as pd
import numpy as np
import torch
import os
import random
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.util import ngrams

In [ ]:
device = "cpu"
checkpoint = "bigcode/starcoder2-7b"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

In [ ]:
def evaluate_different_tokens(df, result_row, reference_row, tokenizer, device):
    """
    Evaluate the performance from different results token lengths.
    Run bleu score on them as well
    Args:
        df (pd.DataFrame): DataFrame containing the results.
        result_row (str): Column name in the DataFrame containing the results.
        reference_row (str): Column name in the DataFrame containing the references.
        tokenizer: Tokenizer for encoding the results.
        device: Device to run the model on.
        
    """
    

    for index, row in df.iterrows():
        result = row[result_row]
        reference = row[reference_row]
        full_results_encoded = tokenizer(result, return_tensors="pt").to(device)
        full_reference_encoded = tokenizer(reference, return_tensors="pt").to(device)
        full_result_tokens = tokenizer.convert_ids_to_tokens(full_results_encoded.input_ids[0])
        full_reference_tokens = tokenizer.convert_ids_to_tokens(full_reference_encoded.input_ids[0])

        result_10_tokens = full_result_tokens[:10]
        reference_10_tokens = full_reference_tokens[:10]
        result_20_tokens = full_result_tokens[:20]
        reference_20_tokens = full_reference_tokens[:20]
        result_30_tokens = full_result_tokens[:30]
        reference_30_tokens = full_reference_tokens[:30]
        result_40_tokens = full_result_tokens[:40]
        reference_40_tokens = full_reference_tokens[:40]

        result_10_string = tokenizer.convert_tokens_to_string(result_10_tokens)
        reference_10_string = tokenizer.convert_tokens_to_string(reference_10_tokens)
        result_20_string = tokenizer.convert_tokens_to_string(result_20_tokens)
        reference_20_string = tokenizer.convert_tokens_to_string(reference_20_tokens)
        result_30_string = tokenizer.convert_tokens_to_string(result_30_tokens)
        reference_30_string = tokenizer.convert_tokens_to_string(reference_30_tokens)
        result_40_string = tokenizer.convert_tokens_to_string(result_40_tokens)
        reference_40_string = tokenizer.convert_tokens_to_string(reference_40_tokens)

        # Calculate BLEU scores
        smooth_fn = SmoothingFunction().method1
        bleu_score_10 = sentence_bleu([reference_10_string], result_10_string, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smooth_fn)
        bleu_score_20 = sentence_bleu([reference_20_string], result_20_string, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smooth_fn)
        bleu_score_30 = sentence_bleu([reference_30_string], result_30_string, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smooth_fn)
        bleu_score_40 = sentence_bleu([reference_40_string], result_40_string, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smooth_fn)

        # load the results into the dataframe
        df.at[index, 'bleu_10'] = bleu_score_10
        df.at[index, 'bleu_20'] = bleu_score_20
        df.at[index, 'bleu_30'] = bleu_score_30
        df.at[index, 'bleu_40'] = bleu_score_40
    return df

In [ ]:
### try with 3b results
dfs_3b_results = []
base_path_3b = "/Users/anonymous_user/Downloads/experiment_results/exp1_membership_inference_attacks_results/starcoder/starcoder_3b"
for file in os.listdir(base_path_3b):
    if file.endswith(".csv"):
        file_path = os.path.join(base_path_3b, file)
        df = pd.read_csv(file_path)
        dfs_3b_results.append(df)
dfs_3b_results = pd.concat(dfs_3b_results, ignore_index=True)

In [ ]:
dfs_3b_results.head()

In [ ]:
import torch
x = torch.rand(5, 3)
print(x)

In [ ]:
torch.cuda.is_available()


In [ ]:
df_trail = evaluate_different_tokens(dfs_3b_results, 'mia_result_50_3b', 'mia_reference_50', tokenizer, device)

In [ ]:
df_trail.head()

In [ ]:
def evaluate_different_tokens(df, result_row, reference_row, tokenizer, device):
    """
    Evaluate BLEU at different token-cutoff lengths (10, 20, 30, 40).
    Args:
        df (pd.DataFrame): DataFrame containing results.
        result_row (str): Column name with the model outputs.
        reference_row (str): Column name with reference texts.
        tokenizer: A Hugging Face tokenizer for encoding.
        device: The device (CPU/GPU) to move the tensors to.
    Returns:
        pd.DataFrame: The original DataFrame with new BLEU columns added.
    """
    smooth_fn = SmoothingFunction().method1

    for index, row in df.iterrows():
        # Ensure these are strings
        result = str(row[result_row]) if pd.notnull(row[result_row]) else ""
        reference = str(row[reference_row]) if pd.notnull(row[reference_row]) else ""

        # Tokenize and get IDs
        full_result_encoded = tokenizer(result, return_tensors="pt").to(device)
        full_reference_encoded = tokenizer(reference, return_tensors="pt").to(device)

        # Convert IDs back to tokens
        full_result_tokens = tokenizer.convert_ids_to_tokens(full_result_encoded.input_ids[0])
        full_reference_tokens = tokenizer.convert_ids_to_tokens(full_reference_encoded.input_ids[0])

        # Truncate tokens to the different lengths
        result_10_tokens = full_result_tokens[:10]
        reference_10_tokens = full_reference_tokens[:10]
        result_20_tokens = full_result_tokens[:20]
        reference_20_tokens = full_reference_tokens[:20]
        result_30_tokens = full_result_tokens[:30]
        reference_30_tokens = full_reference_tokens[:30]
        result_40_tokens = full_result_tokens[:40]
        reference_40_tokens = full_reference_tokens[:40]

        # Calculate BLEU scores. Pass lists of tokens (not single strings).
        bleu_score_10 = sentence_bleu([reference_10_tokens], result_10_tokens,
                                      weights=(0.25, 0.25, 0.25, 0.25),
                                      smoothing_function=smooth_fn)
        bleu_score_20 = sentence_bleu([reference_20_tokens], result_20_tokens,
                                      weights=(0.25, 0.25, 0.25, 0.25),
                                      smoothing_function=smooth_fn)
        bleu_score_30 = sentence_bleu([reference_30_tokens], result_30_tokens,
                                      weights=(0.25, 0.25, 0.25, 0.25),
                                      smoothing_function=smooth_fn)
        bleu_score_40 = sentence_bleu([reference_40_tokens], result_40_tokens,
                                      weights=(0.25, 0.25, 0.25, 0.25),
                                      smoothing_function=smooth_fn)

        # Store scores in the DataFrame
        df.at[index, 'bleu_10'] = bleu_score_10
        df.at[index, 'bleu_20'] = bleu_score_20
        df.at[index, 'bleu_30'] = bleu_score_30
        df.at[index, 'bleu_40'] = bleu_score_40

    return df

In [ ]:
df_trail = evaluate_different_tokens(dfs_3b_results, 'mia_result_50_3b', 'mia_reference_50', tokenizer, device)

In [ ]:
df_trail.head()

In [ ]:
df_trail.shape

In [ ]:
df_trail[df_trail['bleu_10'] ==1].shape

In [ ]:
df_trail[df_trail['bleu_40'] ==1].shape

In [ ]:
df_trail[df_trail['bleu_30'] ==1].shape

In [ ]:
df_trail[df_trail['bleu_20'] ==1].shape

In [ ]:
df_trail[df_trail['bleu_score_4_gram_3b'] ==1].shape

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

def evaluate_tokens_increment_2(df, result_col, reference_col, tokenizer, device):
    """
    Compute BLEU scores at increments of 2 tokens up to 50 for each row.
    Stores columns: bleu_2, bleu_4, bleu_6, ..., bleu_50.
    
    Args:
        df (pd.DataFrame): DataFrame with results/references.
        result_col (str): Column name containing system output text.
        reference_col (str): Column name containing reference text.
        tokenizer: A Hugging Face tokenizer.
        device: 'cpu' or 'cuda'.
        
    Returns:
        pd.DataFrame: The same DataFrame with new BLEU columns for each token cutoff.
    """
    smooth_fn = SmoothingFunction().method1
    token_cutoffs = list(range(2, 51, 2))

    for idx, row in df.iterrows():
        # 1) Ensure text is string (avoid ValueError)
        result_text = str(row[result_col]) if pd.notnull(row[result_col]) else ""
        ref_text = str(row[reference_col]) if pd.notnull(row[reference_col]) else ""

        # 2) Tokenize
        result_encoded = tokenizer(result_text, return_tensors="pt").to(device)
        reference_encoded = tokenizer(ref_text, return_tensors="pt").to(device)
        
        # 3) Convert IDs to tokens
        result_tokens = tokenizer.convert_ids_to_tokens(result_encoded.input_ids[0])
        ref_tokens = tokenizer.convert_ids_to_tokens(reference_encoded.input_ids[0])

        # 4) For each cutoff, compute BLEU
        for cutoff in token_cutoffs:
            truncated_result = result_tokens[:cutoff]
            truncated_ref = ref_tokens[:cutoff]

            bleu = sentence_bleu(
                [truncated_ref],
                truncated_result,
                weights=(0.25, 0.25, 0.25, 0.25),
                smoothing_function=smooth_fn
            )
            df.at[idx, f'bleu_{cutoff}'] = bleu

    return df

def visualize_and_fit_curve(df):
    """
    Takes a DataFrame that already has 'bleu_2', 'bleu_4', ..., 'bleu_50'
    columns. Plots the average BLEU vs. cutoff and also a polynomial fit.
    
    Args:
        df (pd.DataFrame): DataFrame with columns named 'bleu_2', 'bleu_4', ..., 'bleu_50'.
    """
    # 1) Gather token cutoffs (2, 4, 6, ... , 50)
    token_cutoffs = list(range(2, 51, 2))

    # 2) Compute the average BLEU for each cutoff across all rows
    avg_bleus = []
    for cutoff in token_cutoffs:
        col_name = f'bleu_{cutoff}'
        if col_name in df.columns:
            avg_bleus.append(df[col_name].mean())
        else:
            avg_bleus.append(np.nan)

    # 3) Plot the raw data (average BLEU vs. token cutoff)
    plt.figure()
    plt.plot(token_cutoffs, avg_bleus, marker='o', label='Average BLEU')
    plt.title('Average BLEU vs. Token Cutoff')
    plt.xlabel('Number of Tokens')
    plt.ylabel('Average BLEU')
    plt.legend()
    plt.show()

    # 4) Fit a polynomial curve (e.g. degree=2 or 3)
    #    Convert to NumPy arrays for fitting
    x_vals = np.array(token_cutoffs, dtype=float)
    y_vals = np.array(avg_bleus, dtype=float)

    # Remove NaNs (if any) before fitting
    mask = ~np.isnan(y_vals)
    x_vals = x_vals[mask]
    y_vals = y_vals[mask]

    # Fit a 2nd-degree polynomial: y = ax^2 + bx + c
    coeffs = np.polyfit(x_vals, y_vals, deg=2)  # Adjust "deg" as desired
    poly_func = np.poly1d(coeffs)

    # 5) Plot the fitted curve
    #    Create a smooth set of x-values for plotting the curve
    x_smooth = np.linspace(x_vals.min(), x_vals.max(), 200)
    y_smooth = poly_func(x_smooth)

    plt.figure()
    plt.plot(x_vals, y_vals, 'o', label='Average BLEU')  # Original data
    plt.plot(x_smooth, y_smooth, label=f'Poly fit (deg=2)\n{poly_func}')
    plt.title('Polynomial Fit of BLEU vs. Token Cutoff')
    plt.xlabel('Number of Tokens')
    plt.ylabel('Average BLEU')
    plt.legend()
    plt.show()

    print("Polynomial Coefficients (a, b, c):", coeffs)
    print("Polynomial Equation:", poly_func)



In [ ]:
df_trail = evaluate_tokens_increment_2(dfs_3b_results, 'mia_result_50_3b', 'mia_reference_50', tokenizer, device)

visualize_and_fit_curve(df_trail)

In [ ]:
def visualize_perfect_bleu_counts(df):
    """
    For each token cutoff (2, 4, ..., 50), count how many samples 
    have a perfect BLEU score (1.0). Then plot and fit a line.
    
    The DataFrame is assumed to have columns named 'bleu_2', 'bleu_4',
    ..., 'bleu_50' from a prior function call.
    """
    import numpy as np
    import matplotlib.pyplot as plt

    # 1) Identify the token cutoffs we used
    token_cutoffs = list(range(2, 51, 2))

    # 2) Count how many rows are perfect (BLEU == 1.0) for each cutoff
    perfect_counts = []
    for cutoff in token_cutoffs:
        col_name = f'bleu_{cutoff}'
        if col_name in df.columns:
            perfect_count = (df[col_name] == 1.0).sum()
            perfect_counts.append(perfect_count)
        else:
            # If the column doesn't exist, append NaN or 0
            perfect_counts.append(np.nan)

    # 3) Convert to NumPy arrays
    x_vals = np.array(token_cutoffs, dtype=float)
    y_vals = np.array(perfect_counts, dtype=float)

    # Remove NaNs (if any) for proper fitting
    valid_mask = ~np.isnan(y_vals)
    x_vals = x_vals[valid_mask]
    y_vals = y_vals[valid_mask]

    # 4) Fit a line (1st-degree polynomial: y = m*x + b)
    m, b = np.polyfit(x_vals, y_vals, deg=1)  # slope = m, intercept = b

    # 5) Create a smooth set of x-values for the line
    x_smooth = np.linspace(x_vals.min(), x_vals.max(), 200)
    y_smooth = m * x_smooth + b

    # 6) Plot the data points
    plt.figure()
    plt.plot(x_vals, y_vals, 'o', label='Perfect BLEU counts')

    # 7) Plot the fitted line
    plt.plot(x_smooth, y_smooth, label=f'Line Fit (y = {m:.3f}x + {b:.3f})')

    plt.title('# of Perfect BLEU Scores vs. Token Cutoff')
    plt.xlabel('Token Cutoff')
    plt.ylabel('Count of Perfect BLEU = 1.0')
    plt.legend()
    plt.show()

    print("Fitted line: y = {:.4f}*x + {:.4f}".format(m, b))
    print(f"Where x is token cutoff (2..50).")


In [ ]:
visualize_perfect_bleu_counts(df_trail)

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

def visualize_perfect_bleu_seaborn(df, poly_order=2):
    """
    Compute the number of perfect BLEU score samples (BLEU==1.0) for
    token cutoffs from 2 to 50 (step=2). Then, fit a polynomial regression 
    (default degree=2) using Seaborn's regplot and visualize the data with a 
    paper-grade, beautiful graph.
    
    Args:
        df (pd.DataFrame): DataFrame that includes columns 'bleu_2', 'bleu_4', ... 'bleu_50'
        poly_order (int): The order (degree) of the polynomial fit.
        
    Returns:
        None. Displays the plot and prints the fitted polynomial coefficients.
    """
    # 1. Build lists of token cutoffs and the corresponding perfect counts.
    token_cutoffs = list(range(2, 51, 2))
    perfect_counts = []
    
    for cutoff in token_cutoffs:
        col_name = f'bleu_{cutoff}'
        # Count rows that have a perfect BLEU score
        if col_name in df.columns:
            perfect_count = (df[col_name] == 1.0).sum()
            perfect_counts.append(perfect_count)
        else:
            perfect_counts.append(np.nan)
    
    # Create a DataFrame to hold our counts
    data = pd.DataFrame({
        'TokenCutoff': token_cutoffs,
        'PerfectBLEUCount': perfect_counts
    })
    
    # 2. Compute the polynomial coefficients using np.polyfit for display purposes
    valid_mask = ~data['PerfectBLEUCount'].isna()
    coeffs = np.polyfit(data['TokenCutoff'][valid_mask], data['PerfectBLEUCount'][valid_mask], poly_order)
    poly_eq = " + ".join([f"{coef:.3f}x^{poly_order-i}" for i, coef in enumerate(coeffs)])
    
    # 3. Set up Seaborn aesthetics for a paper-grade graph.
    sns.set_theme(style="whitegrid", context="paper")
    plt.figure(figsize=(8, 6))
    
    # 4. Create the plot using regplot for polynomial fit.
    ax = sns.regplot(
        x='TokenCutoff', 
        y='PerfectBLEUCount', 
        data=data, 
        order=poly_order, 
        ci=95, 
        scatter_kws={'s': 70, 'color': 'navy'},
        line_kws={'color': 'red', 'lw': 2}
    )
    
    ax.set_title('Perfect BLEU Count vs. Token Cutoff\nPolynomial Fit (Degree={})'.format(poly_order), fontsize=14)
    ax.set_xlabel('Token Cutoff (Number of Tokens)', fontsize=12)
    ax.set_ylabel('Count of Perfect BLEU Scores (1.0)', fontsize=12)
    
    # Annotate the plot with the polynomial equation.
    plt.text(0.05, 0.95, f'Fit: {poly_eq}', transform=ax.transAxes,
             fontsize=12, verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.5))
    
    plt.tight_layout()
    plt.show()
    
    print("Fitted Polynomial Coefficients (highest degree first):", coeffs)

# Example usage:
# Assuming `df` already has the BLEU columns (bleu_2, bleu_4, ..., bleu_50)
# visualize_perfect_bleu_seaborn(df, poly_order=2)


In [ ]:
visualize_perfect_bleu_seaborn(df_trail, poly_order=2)

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

def visualize_perfect_bleu_seaborn_no_first_point(df, poly_order=2):
    """
    Compute the number of perfect BLEU score samples (BLEU==1.0) for
    token cutoffs from 2 to 50 (step=2). Then, remove the first point 
    (token cutoff = 2) before fitting a polynomial regression using Seaborn.
    
    Args:
        df (pd.DataFrame): DataFrame that includes columns 'bleu_2', 'bleu_4', ... 'bleu_50'
        poly_order (int): Degree of the polynomial fit.
        
    Returns:
        None. Displays the plot and prints the fitted polynomial coefficients.
    """
    # Build lists of token cutoffs and corresponding perfect BLEU counts.
    token_cutoffs = list(range(2, 51, 2))
    perfect_counts = []
    
    for cutoff in token_cutoffs:
        col_name = f'bleu_{cutoff}'
        # Count rows that have a perfect BLEU score (i.e., exactly 1.0)
        if col_name in df.columns:
            perfect_count = (df[col_name] == 1.0).sum()
            perfect_counts.append(perfect_count)
        else:
            perfect_counts.append(np.nan)
    
    # Create a DataFrame holding our counts.
    data = pd.DataFrame({
        'TokenCutoff': token_cutoffs,
        'PerfectBLEUCount': perfect_counts
    })
    
    # Remove the first data point (TokenCutoff = 2)
    data = data.iloc[1:]
    
    # Compute polynomial coefficients using np.polyfit for display.
    valid_mask = ~data['PerfectBLEUCount'].isna()
    coeffs = np.polyfit(data['TokenCutoff'][valid_mask], data['PerfectBLEUCount'][valid_mask], poly_order)
    poly_eq = " + ".join([f"{coef:.3f}x^{poly_order-i}" for i, coef in enumerate(coeffs)])
    
    # Set up Seaborn aesthetics for a publication-quality graph.
    sns.set_theme(style="whitegrid", context="paper")
    plt.figure(figsize=(8, 6))
    
    # Use Seaborn's regplot to plot data and polynomial fit.
    ax = sns.regplot(
        x='TokenCutoff', 
        y='PerfectBLEUCount', 
        data=data, 
        order=poly_order, 
        ci=95, 
        scatter_kws={'s': 70, 'color': 'navy'},
        line_kws={'color': 'red', 'lw': 2}
    )
    
    ax.set_title('Perfect BLEU Count vs. Token Cutoff\nPolynomial Fit (Degree={})'.format(poly_order), fontsize=14)
    ax.set_xlabel('Token Cutoff (Number of Tokens)', fontsize=12)
    ax.set_ylabel('Count of Perfect BLEU Scores (1.0)', fontsize=12)
    
    # Annotate the plot with the polynomial equation.
    plt.text(0.05, 0.95, f'Fit: {poly_eq}', transform=ax.transAxes,
             fontsize=12, verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.5))
    
    plt.tight_layout()
    plt.show()
    
    print("Fitted Polynomial Coefficients (highest degree first):", coeffs)

# Example usage:
# Assuming `df` has columns: bleu_2, bleu_4, ..., bleu_50
# visualize_perfect_bleu_seaborn_no_first_point(df, poly_order=2)


In [ ]:
visualize_perfect_bleu_seaborn_no_first_point(df_trail, poly_order=2)

In [ ]:
path_all_3b = "/Users/anonymous_user/Downloads/experiment_results/exp1_membership_inference_attacks_results/starcoder/starcoder_3b"

In [ ]:
dfs_3b_results = []

In [ ]:
for file in os.listdir(path_all_3b):
    if file.endswith(".csv"):
        file_path = os.path.join(path_all_3b, file)
        df = pd.read_csv(file_path)
        dfs_3b_results.append(df)
dfs_3b_results = pd.concat(dfs_3b_results, ignore_index=True)
dfs_3b_results.shape

In [ ]:
df_results = evaluate_tokens_increment_2(dfs_3b_results, 'mia_result_50_3b', 'mia_reference_50', tokenizer, device)
df_results.shape

In [ ]:
visualize_perfect_bleu_seaborn_no_first_point(df_results, poly_order=2)

In [ ]:
dfs_3b_results.head()

In [ ]:
df = pd.read_csv("/Users/anonymous_user/Downloads/experiment_results/exp1_membership_inference_attacks_results/llama/result/output/repo_path_size_copies_contents_000000000010.csv")

In [ ]:
df.head()

In [ ]:
### import libraries

import pandas as pd
import numpy as np
import torch
import os
import random
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.util import ngrams
import os
import glob
import pandas as pd
import torch
from transformers import AutoTokenizer

# Provided function  (ensure this is defined, or import if in another module)
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
